In [1]:
# ── TunnelScope ML — Stage 1 setup ────────────────────────────────
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold, RepeatedKFold

SEED = 42
np.random.seed(SEED)

DATA = Path("../data")
TRAIN = DATA / "synthetic_ipsec_dataset_train.csv"
TEST  = DATA / "synthetic_ipsec_dataset_test.csv"

train = pd.read_csv(TRAIN)
test  = pd.read_csv(TEST)

# the 10 features — exact names, must match the Spring Boot DTOs
CAT_FEATURES = ["mode", "encryption_algo", "dh_group", "pfs", "ip_version"]
NUM_FEATURES = ["avg_packet_size_bytes", "packet_rate_per_sec",
                "session_duration_sec", "burstiness_index",
                "ike_handshake_time_ms"]
FEATURES = CAT_FEATURES + NUM_FEATURES

# behaviour-only subset — traffic_type model uses ONLY these
BEHAVIOUR_FEATURES = NUM_FEATURES

TARGETS = ["traffic_type", "security_verdict", "risk_score"]

# one shared fold generator — every candidate sees the SAME splits
CV_CLF = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
CV_REG = RepeatedKFold(n_splits=5, n_repeats=5, random_state=SEED)

print(f"train {train.shape}   test {test.shape}")
print(f"nulls: train={train.isna().sum().sum()}  test={test.isna().sum().sum()}")
print(f"\ntraffic_type classes ({train.traffic_type.nunique()}):")
print(train.traffic_type.value_counts().to_string())
print(f"\nsecurity_verdict:")
print(train.security_verdict.value_counts().to_string())

print("\n--- LEAK CHECK: risk_score bands per verdict ---")
bands = train.groupby("security_verdict")["risk_score"].agg(["min", "max"])
print(bands.to_string())
s_max = bands.loc["Strong", "max"]; m_min = bands.loc["Medium", "min"]
m_max = bands.loc["Medium", "max"]; w_min = bands.loc["Weak", "min"]
if s_max > m_min and m_max > w_min:
    print("OK  -> bands overlap, risk_score is not a perfect proxy")
else:
    print("WARN-> bands are disjoint. You are on the OLD csv. Replace it.")

train (9000, 14)   test (1200, 14)
nulls: train=0  test=0

traffic_type classes (8):
traffic_type
icmp               1213
web_browsing       1175
whatsapp_msg       1138
file_transfer      1118
video_streaming    1112
email              1099
dns_query          1076
voip               1069

security_verdict:
security_verdict
Medium    3550
Weak      2849
Strong    2601

--- LEAK CHECK: risk_score bands per verdict ---
                  min   max
security_verdict           
Medium            2.2   7.4
Strong            0.3   4.0
Weak              5.8  10.0
OK  -> bands overlap, risk_score is not a perfect proxy


In [5]:
# ── Cell 2: preprocessing ────────────────────────────────────────
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline


def make_preprocessor(cat_features, num_features):
    """Text -> one-hot columns, numbers -> standardised.

    Never fit this directly. Always put it inside a Pipeline so that
    fitting happens on the training fold only.
    """
    transformers = []

    if cat_features:
        transformers.append((
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",   # unseen category -> all zeros, no crash
                sparse_output=False,       # dense: skl2onnx exports this cleanly
            ),
            cat_features,
        ))

    if num_features:
        transformers.append((
            "num",
            StandardScaler(),
            num_features,
        ))

    return ColumnTransformer(transformers, remainder="drop")


PREP_FULL      = make_preprocessor(CAT_FEATURES, NUM_FEATURES)      # 10 features
PREP_BEHAVIOUR = make_preprocessor([], BEHAVIOUR_FEATURES)          # 5 numeric only


def build(estimator, prep="full"):
    """Wrap any estimator into a leak-safe pipeline."""
    base = PREP_FULL if prep == "full" else PREP_BEHAVIOUR
    return Pipeline([("prep", clone(base)), ("model", estimator)])


# --- sanity check: what shape does the model actually receive? ---
_full = clone(PREP_FULL).fit(train[FEATURES])
_beh  = clone(PREP_BEHAVIOUR).fit(train[BEHAVIOUR_FEATURES])
print("full pipeline      ->", _full.transform(train[FEATURES]).shape[1], "columns")
print("behaviour pipeline ->", _beh.transform(train[BEHAVIOUR_FEATURES]).shape[1], "columns")
print()
print("expanded categorical names:")
print(list(_full.named_transformers_["cat"].get_feature_names_out(CAT_FEATURES)))

full pipeline      -> 25 columns
behaviour pipeline -> 5 columns

expanded categorical names:
['mode_Transport', 'mode_Tunnel', 'encryption_algo_3DES', 'encryption_algo_AES-128', 'encryption_algo_AES-192', 'encryption_algo_AES-256', 'encryption_algo_AES-256-GCM', 'encryption_algo_DES', 'dh_group_modp1024(2)', 'dh_group_modp1536(5)', 'dh_group_modp2048(14)', 'dh_group_modp3072(15)', 'dh_group_modp4096(16)', 'dh_group_modp6144(17)', 'dh_group_modp768(1)', 'dh_group_modp8192(21)', 'pfs_Off', 'pfs_On', 'ip_version_IPv4', 'ip_version_IPv6']


In [6]:
# Waise clone yahan zaroori kyun hai, ye samajh lo — baad mein kaam aayega. PREP_FULL ek hi object hai. Agar hum wahi object seedha har pipeline mein daal dein, to teeno-chaaron models ek hi preprocessor share karenge. Ek model fit hoga to uska state doosre mein chala jayega. clone() har baar ek fresh unfitted copy banata hai, same settings ke saath. Isliye har candidate ko apna saaf preprocessor milta hai.

In [7]:
# ── Cell 3: evaluation harness ───────────────────────────────────
import time
import warnings
from sklearn.model_selection import cross_validate

RESULTS = []   # every evaluated candidate lands here


CLF_SCORING = {
    "f1_macro": "f1_macro",
    "accuracy": "accuracy",
}

REG_SCORING = {
    "mae": "neg_mean_absolute_error",
    "r2":  "r2",
}


def evaluate(name, estimator, X, y, task="clf", prep="full", note=""):
    """Run one candidate through the shared folds and record the result."""
    pipe = build(estimator, prep=prep)
    cv = CV_CLF if task == "clf" else CV_REG
    scoring = CLF_SCORING if task == "clf" else REG_SCORING

    t0 = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        cv_out = cross_validate(
            pipe, X, y,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
            error_score="raise",   # fail loudly, never silently score NaN
        )
    elapsed = time.perf_counter() - t0

    row = {"model": name, "task": task, "note": note, "seconds": round(elapsed, 1)}

    for key in scoring:
        vals = cv_out[f"test_{key}"]
        if key == "mae":
            vals = -vals          # sklearn returns negative MAE
        row[f"{key}_mean"] = round(float(vals.mean()), 4)
        row[f"{key}_std"] = round(float(vals.std()), 4)

    # conservative ranking number: mean minus one std
    primary = "f1_macro" if task == "clf" else "mae"
    if task == "clf":
        row["floor"] = round(row["f1_macro_mean"] - row["f1_macro_std"], 4)
    else:
        row["floor"] = round(row["mae_mean"] + row["mae_std"], 4)

    RESULTS.append(row)

    if task == "clf":
        print(f"{name:22s}  F1 {row['f1_macro_mean']:.4f} ± {row['f1_macro_std']:.4f}"
              f"   acc {row['accuracy_mean']:.4f}   floor {row['floor']:.4f}"
              f"   [{row['seconds']}s]")
    else:
        print(f"{name:22s}  MAE {row['mae_mean']:.4f} ± {row['mae_std']:.4f}"
              f"   R2 {row['r2_mean']:.4f}   ceiling {row['floor']:.4f}"
              f"   [{row['seconds']}s]")
    return row


def leaderboard(task="clf", target=None):
    """Show recorded results, best first."""
    df = pd.DataFrame([r for r in RESULTS if r["task"] == task])
    if target:
        df = df[df.note == target]
    if df.empty:
        return df
    if task == "clf":
        return df.sort_values("floor", ascending=False).reset_index(drop=True)
    return df.sort_values("floor", ascending=True).reset_index(drop=True)


print("harness ready")
print(f"classification folds : {CV_CLF.get_n_splits()} fits per candidate")
print(f"regression folds     : {CV_REG.get_n_splits()} fits per candidate")

harness ready
classification folds : 25 fits per candidate
regression folds     : 25 fits per candidate


In [8]:
# Cell 3 ek judge bana raha hai. Model nahi.

# Cricket ki pitch samajh lo — batsman abhi nahi aaye, bas pitch aur scorer set kar rahe hain, taaki sab same conditions mein khelein.

# Isme 4 cheezein hain:

# RESULTS = [] — khaali list. Har model ka result isme jama hoga. Isliye ye cell sirf ek baar chalana, dobara chalaya to purana sab mit jayega.

# evaluate() — asli kaam. Ek model do, ye use pipeline mein lapetega, 25 baar train karega (5 folds × 5 repeats), 25 scores ka average aur std nikalega, list mein daal dega, aur ek line print kar dega. 25 baar isliye kyunki ek baar mein score kismat pe depend karta hai — kaunsi rows test mein gayi. 25 baar se woh factor nikal jaata hai.

# floor — ranking isi pe hogi. mean − std. Matlab "bura din aaya to kam se kam itna milega". A = 0.94 ± 0.01 → floor 0.93. B = 0.95 ± 0.04 → floor 0.91. Mean se B jeeta, floor se A jeeta. Humein bharosemand model chahiye, lucky nahi.

# leaderboard() — bas list ko sort karke table dikhata hai.

# Plus teen settings: error_score="raise" (model crash ho to pata chale, chup-chaap NaN na aaye), n_jobs=-1 (saare CPU cores, warna SVM slow), warnings off (screen saaf rahe).

In [10]:
# ── Cell 4: traffic_type — fast candidates ───────────────────────
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                           QuadraticDiscriminantAnalysis)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              HistGradientBoostingClassifier, AdaBoostClassifier)

# is cell ko dobara chalane par purane traffic_type results hat jaayenge
RESULTS[:] = [r for r in RESULTS if r["note"] != "traffic_type"]

# behaviour features ONLY — config columns ka traffic type se koi taluk nahi
X_tt = train[BEHAVIOUR_FEATURES]
y_tt = train["traffic_type"]

print(f"X {X_tt.shape}   y {y_tt.nunique()} classes")
print(f"majority-class baseline = {y_tt.value_counts(normalize=True).max():.4f}\n")

FAST_CLF = {
    "Dummy":        DummyClassifier(strategy="most_frequent"),
    "LogisticReg":  LogisticRegression(max_iter=2000, random_state=SEED),
    "RidgeClf":     RidgeClassifier(random_state=SEED),
    "SGD":          SGDClassifier(random_state=SEED),
    "LDA":          LinearDiscriminantAnalysis(),
    # reg_param zaroori hai: dns_query ki covariance matrix singular hai
    "QDA":          QuadraticDiscriminantAnalysis(reg_param=0.1),
    "GaussianNB":   GaussianNB(),
    "kNN":          KNeighborsClassifier(n_neighbors=5),
    "DecisionTree": DecisionTreeClassifier(random_state=SEED),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=1),
    "ExtraTrees":   ExtraTreesClassifier(n_estimators=300, random_state=SEED, n_jobs=1),
    "HistGB":       HistGradientBoostingClassifier(random_state=SEED),
    "AdaBoost":     AdaBoostClassifier(random_state=SEED),
}

for name, est in FAST_CLF.items():
    evaluate(name, est, X_tt, y_tt, task="clf", prep="behaviour", note="traffic_type")

print(f"\ntraffic_type candidates recorded: "
      f"{len([r for r in RESULTS if r['note'] == 'traffic_type'])}")

X (9000, 5)   y 8 classes
majority-class baseline = 0.1348

Dummy                   F1 0.0297 ± 0.0001   acc 0.1348   floor 0.0296   [4.0s]
LogisticReg             F1 0.9091 ± 0.0061   acc 0.9088   floor 0.9030   [0.8s]
RidgeClf                F1 0.5644 ± 0.0076   acc 0.6418   floor 0.5568   [0.2s]
SGD                     F1 0.7991 ± 0.0306   acc 0.8138   floor 0.7685   [0.8s]
LDA                     F1 0.8686 ± 0.0071   acc 0.8706   floor 0.8615   [0.3s]
QDA                     F1 0.8474 ± 0.0075   acc 0.8496   floor 0.8399   [0.2s]
GaussianNB              F1 0.9462 ± 0.0040   acc 0.9460   floor 0.9422   [0.1s]
kNN                     F1 0.9109 ± 0.0064   acc 0.9107   floor 0.9045   [0.3s]
DecisionTree            F1 0.9203 ± 0.0057   acc 0.9200   floor 0.9146   [0.4s]
RandomForest            F1 0.9480 ± 0.0045   acc 0.9478   floor 0.9435   [8.6s]
ExtraTrees              F1 0.9480 ± 0.0047   acc 0.9478   floor 0.9433   [6.4s]
HistGB                  F1 0.9435 ± 0.0060   acc 0.9434   fl

In [11]:
# Pehla asli model-training cell. Target = traffic_type (8 classes: voip, dns_query, icmp, etc.).

# Teen hisse:

# Data chuna — X_tt mein sirf 5 behaviour features (packet size, rate, duration, burstiness, handshake time). Config columns nahi, kyunki encryption algo se ye nahi pata chalta ki traffic VoIP hai ya DNS.
# 13 models ki list — FAST_CLF dictionary. Linear wale, tree wale, distance wale, boosting wale. Sab fast hain, isliye ek saath.
# Loop — har model evaluate() mein jaata hai, 25 baar train hota hai, ek line print karta hai. Sab kuch RESULTS mein jama.

# Dummy sabse pehle kyun: woh kuch seekhta nahi, bas hamesha sabse common class bol deta hai (~13%). Woh baseline hai. Koi bhi model uske aas-paas aaya = woh kuch seekh hi nahi raha.

# prep="behaviour" = 5 features wali pipeline. n_jobs=1 RF ke andar isliye kyunki bahar already saare cores chal rahe hain, dono jagah karoge to ladenge.

# Aage kya aayega, taaki picture clear rahe:

# Cell 5 — traffic_type ke slow models (SVM, MLP, GradientBoosting). Alag isliye ki ye time lenge.
# Cell 6 — traffic_type ki leaderboard + confusion matrix. Yahan dekhenge dns_query aur icmp confuse ho rahe hain ya nahi.
# Cell 7-9 — wahi teen cheezein security_verdict ke liye, par full 10 features ke saath.
# Cell 10-12 — risk_score regression. Alag models (Ridge, SVR, regressors).
# Cell 13 — teeno tables saath mein, top 4-5 shortlist.

# Uske baad Stage 2 — tuning, ONNX export, demo CSVs pe test.

In [13]:
# ── Cell 5: traffic_type — slow candidates ───────────────────────
from sklearn.svm import SVC, LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier

RESULTS[:] = [r for r in RESULTS if r["note"] != "traffic_type_slow"]

SLOW_CLF = {
    "SVM-rbf":      SVC(kernel="rbf", random_state=SEED),
    "SVM-linear":   LinearSVC(max_iter=5000, random_state=SEED),
    "SVM-poly":     SVC(kernel="poly", degree=3, random_state=SEED),
    "MLP":          MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=600,
                                  random_state=SEED),
    "GradientBoost": GradientBoostingClassifier(random_state=SEED),
}

for name, est in SLOW_CLF.items():
    evaluate(name, est, X_tt, y_tt, task="clf", prep="behaviour",
             note="traffic_type_slow")

SVM-rbf                 F1 0.9201 ± 0.0051   acc 0.9199   floor 0.9150   [2.3s]
SVM-linear              F1 0.8690 ± 0.0067   acc 0.8701   floor 0.8623   [0.3s]
SVM-poly                F1 0.9085 ± 0.0070   acc 0.9082   floor 0.9015   [3.9s]
MLP                     F1 0.9396 ± 0.0040   acc 0.9394   floor 0.9356   [81.8s]
GradientBoost           F1 0.9469 ± 0.0048   acc 0.9468   floor 0.9421   [46.3s]


In [14]:
# Cell 5 kya kar raha hai:

# Cell 4 jaisa hi kaam, bas 5 slow models. Same data, same folds, same judge — sirf candidates naye.

# Alag cell isliye kyunki ye time lenge. Sab ek saath daalte to 10 minute khaali screen dekhni padti.

# Models:

# SVM-rbf — curved boundary
# SVM-linear — seedhi line, LogisticReg (0.909) ke aas-paas aana chahiye
# SVM-poly — degree-3 curve
# MLP — chhota neural net, 64 → 32 neurons
# GradientBoost — trees ek-ek karke, har naya pichhle ki galti sudhaarta hua

In [15]:
# ── Cell 6: traffic_type — leaderboard + confusion matrix ────────
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report

tt = pd.DataFrame([r for r in RESULTS if r["note"].startswith("traffic_type")])
tt = tt.sort_values("floor", ascending=False).reset_index(drop=True)
print(tt[["model", "f1_macro_mean", "f1_macro_std", "accuracy_mean",
          "floor", "seconds"]].to_string())

BEST_TT = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)

pred = cross_val_predict(
    build(BEST_TT, prep="behaviour"), X_tt, y_tt,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    n_jobs=-1,
)

labels = sorted(y_tt.unique())
cm = pd.DataFrame(
    confusion_matrix(y_tt, pred, labels=labels),
    index=[f"true_{l}" for l in labels],
    columns=[f"pred_{l}" for l in labels],
)
print("\n--- confusion matrix (RandomForest) ---")
print(cm.to_string())

print("\n--- per-class report ---")
print(classification_report(y_tt, pred, digits=3))

            model  f1_macro_mean  f1_macro_std  accuracy_mean   floor  seconds
0    RandomForest         0.9480        0.0045         0.9478  0.9435      8.6
1      ExtraTrees         0.9480        0.0047         0.9478  0.9433      6.4
2      GaussianNB         0.9462        0.0040         0.9460  0.9422      0.1
3   GradientBoost         0.9469        0.0048         0.9468  0.9421     46.3
4          HistGB         0.9435        0.0060         0.9434  0.9375     12.4
5             MLP         0.9396        0.0040         0.9394  0.9356     81.8
6         SVM-rbf         0.9201        0.0051         0.9199  0.9150      2.3
7    DecisionTree         0.9203        0.0057         0.9200  0.9146      0.4
8             kNN         0.9109        0.0064         0.9107  0.9045      0.3
9     LogisticReg         0.9091        0.0061         0.9088  0.9030      0.8
10       SVM-poly         0.9085        0.0070         0.9082  0.9015      3.9
11     SVM-linear         0.8690        0.0067      

In [16]:
# Do cheezein. Pehle poori leaderboard ek table mein — Cell 4 aur 5 dono ke results jodkar, floor pe sort. Fir top model ka confusion matrix.

# Confusion matrix isliye zaroori hai ki 0.948 ka matlab hai 5% galtiyan — par kaunsi galtiyan? Agar model dns_query ko icmp samajh raha hai to woh ek samajhne layak confusion hai (dono chhote packets, kam rate). Agar voip ko file_transfer samajh raha hai to kuch gadbad hai. Ye number se nahi, matrix se pata chalega.

# Iske liye cross_val_predict use karunga — har row ka prediction tab liya jaata hai jab woh test fold mein ho, yaani honest predictions.

In [17]:
# ── Cell 6b: top confusions, compact ─────────────────────────────
cmv = confusion_matrix(y_tt, pred, labels=labels)

pairs = []
for i, t in enumerate(labels):
    for j, p in enumerate(labels):
        if i != j and cmv[i, j] >= 20:
            pairs.append({"true": t, "predicted": p, "count": int(cmv[i, j]),
                          "pct_of_class": round(100 * cmv[i, j] / cmv[i].sum(), 1)})

print("--- confusions >= 20 ---")
print(pd.DataFrame(pairs).sort_values("count", ascending=False).to_string(index=False))

recall = pd.DataFrame({
    "class": labels,
    "support": cmv.sum(axis=1),
    "correct": cmv.diagonal(),
})
recall["recall"] = (recall.correct / recall.support).round(3)
print("\n--- per-class recall (worst first) ---")
print(recall.sort_values("recall").to_string(index=False))

--- confusions >= 20 ---
           true       predicted  count  pct_of_class
   web_browsing           email     82           7.0
      dns_query            icmp     77           7.2
          email    web_browsing     76           6.9
           icmp       dns_query     60           4.9
video_streaming   file_transfer     49           4.4
          email    whatsapp_msg     45           4.1
  file_transfer video_streaming     43           3.8

--- per-class recall (worst first) ---
          class  support  correct  recall
          email     1099      978   0.890
      dns_query     1076      989   0.919
   web_browsing     1175     1088   0.926
           icmp     1213     1153   0.951
video_streaming     1112     1063   0.956
  file_transfer     1118     1075   0.962
   whatsapp_msg     1138     1111   0.976
           voip     1069     1069   1.000


In [18]:
# Cell 6b kya kar raha hai: confusion matrix se sirf woh pairs nikaal raha hai jahan 20 se zyada galtiyan hui. Poori 8×8 grid ki jagah ek chhoti ranked list — isse truncate nahi hoga aur padhna aasan hoga. Saath mein per-class recall bhi, taaki pata chale kaunsi class sabse zyada suffer kar rahi hai.

In [19]:
# ── Cell 7: security_verdict — setup + diagnostics ───────────────
X_sv = train[FEATURES]
y_sv = train["security_verdict"]

print(f"X {X_sv.shape}   classes: {dict(y_sv.value_counts())}")
print(f"majority-class baseline = {y_sv.value_counts(normalize=True).max():.4f}\n")

combos = train.groupby(CAT_FEATURES)["security_verdict"].nunique()
ambiguous = train.groupby(CAT_FEATURES).filter(lambda d: d.security_verdict.nunique() > 1)
print(f"distinct config combos      : {len(combos)}")
print(f"combos with >1 verdict      : {(combos > 1).sum()}")
print(f"rows in ambiguous combos    : {len(ambiguous)} ({100*len(ambiguous)/len(train):.1f}%)")
print("\n-> config alone cannot reach 100%. ML has real work here.\n")

print("--- verdict vs encryption_algo ---")
print(pd.crosstab(train.security_verdict, train.encryption_algo).to_string())

X (9000, 10)   classes: {'Medium': np.int64(3550), 'Weak': np.int64(2849), 'Strong': np.int64(2601)}
majority-class baseline = 0.3944

distinct config combos      : 384
combos with >1 verdict      : 91
rows in ambiguous combos    : 2228 (24.8%)

-> config alone cannot reach 100%. ML has real work here.

--- verdict vs encryption_algo ---
encryption_algo   3DES  AES-128  AES-192  AES-256  AES-256-GCM  DES
security_verdict                                                   
Medium             699      622      575      518          557  579
Strong              44      387      559      793          817    1
Weak               770      546      373      179           48  933


In [20]:
# Cell 7 kya kar raha hai: ab doosra target — security_verdict (Strong/Medium/Weak). Yahan teen cheezein badal rahi hain.

# Ek, ab poore 10 features milenge, sirf behaviour nahi — kyunki verdict ka asli source config hai (encryption algo, dh group, pfs). Isliye prep="full".

# Do, isme ek diagnostic pehle chala raha hoon. Purane dataset mein verdict config se 100% determined tha, yaani ML ki zaroorat hi nahi thi. Naye dataset mein 25% rows ambiguous hain. Pehle woh confirm karte hain, fir models.

# Teen, ek behaviour-only baseline bhi chala raha hoon jaan-boojhkar. Expectation ye hai ki woh bura perform karega — kyunki packet size se ye nahi pata chalta ki encryption kitni majboot hai. Agar woh sach mein bura aaya, to woh proof hai ki architecture ne concerns theek se alag kiye hain. Handoff mein bhi yahi baat likhi hai.

In [22]:
# ── Cell 8: security_verdict — fast candidates ───────────────────
RESULTS[:] = [r for r in RESULTS if not r["note"].startswith("security_verdict")]

SV_FAST = {
    "Dummy":        DummyClassifier(strategy="most_frequent"),
    "LogisticReg":  LogisticRegression(max_iter=2000, random_state=SEED),
    "RidgeClf":     RidgeClassifier(random_state=SEED),
    "SGD":          SGDClassifier(random_state=SEED),
    "LDA":          LinearDiscriminantAnalysis(),
    "QDA":          QuadraticDiscriminantAnalysis(reg_param=0.1),
    "GaussianNB":   GaussianNB(),
    "kNN":          KNeighborsClassifier(n_neighbors=5),
    "DecisionTree": DecisionTreeClassifier(random_state=SEED),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=1),
    "ExtraTrees":   ExtraTreesClassifier(n_estimators=300, random_state=SEED, n_jobs=1),
    "HistGB":       HistGradientBoostingClassifier(random_state=SEED),
    "AdaBoost":     AdaBoostClassifier(random_state=SEED),
}

for name, est in SV_FAST.items():
    evaluate(name, est, X_sv, y_sv, task="clf", prep="full", note="security_verdict")

print("\n--- control: behaviour features only (config hataake) ---")
evaluate("RF-behaviour-only", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=1),
         train[BEHAVIOUR_FEATURES], y_sv, task="clf", prep="behaviour",
         note="security_verdict_control")

Dummy                   F1 0.1886 ± 0.0000   acc 0.3944   floor 0.1886   [0.3s]
LogisticReg             F1 0.9045 ± 0.0058   acc 0.9017   floor 0.8987   [0.5s]
RidgeClf                F1 0.8020 ± 0.0076   acc 0.7983   floor 0.7944   [0.3s]
SGD                     F1 0.8798 ± 0.0112   acc 0.8775   floor 0.8686   [0.6s]
LDA                     F1 0.9048 ± 0.0059   acc 0.9019   floor 0.8989   [0.4s]
QDA                     F1 0.8848 ± 0.0067   acc 0.8832   floor 0.8781   [0.4s]
GaussianNB              F1 0.7171 ± 0.0305   acc 0.7364   floor 0.6866   [1.6s]
kNN                     F1 0.8410 ± 0.0069   acc 0.8374   floor 0.8341   [3.5s]
DecisionTree            F1 0.9029 ± 0.0076   acc 0.9001   floor 0.8953   [0.3s]
RandomForest            F1 0.9043 ± 0.0062   acc 0.9015   floor 0.8981   [9.3s]
ExtraTrees              F1 0.9036 ± 0.0071   acc 0.9008   floor 0.8965   [8.4s]
HistGB                  F1 0.9022 ± 0.0063   acc 0.8993   floor 0.8959   [10.4s]
AdaBoost                F1 0.6964 ± 0.0

{'model': 'RF-behaviour-only',
 'task': 'clf',
 'note': 'security_verdict_control',
 'seconds': 50.6,
 'f1_macro_mean': 0.3306,
 'f1_macro_std': 0.0103,
 'accuracy_mean': 0.3548,
 'accuracy_std': 0.0097,
 'floor': 0.3203}

In [23]:
# Cell 8 kya kar raha hai: wahi 13 fast models, par ab security_verdict pe aur poore 10 features ke saath (prep="full").

# Ek extra cheez isme hai — sabse aakhir mein ek behaviour-only baseline chala raha hoon. Yaani wahi verdict predict karna par sirf packet size, rate, burstiness se, config columns hataakar. Expectation: ye bura perform karega, baseline 0.39 ke aas-paas.

# Agar woh sach mein bura aaya, to woh negative result actually valuable hai — woh saabit karta hai ki traffic behaviour se security strength predict nahi hoti, aur isliye tumhara architecture (rules engine config dekhta hai, ML behaviour dekhta hai) sahi tarah se separated hai. Handoff mein yahi baat likhi hai, ab naye dataset pe dobara confirm ho jayegi.

In [24]:
# ── Cell 9: security_verdict — slow candidates + rule extraction ──
RESULTS[:] = [r for r in RESULTS if r["note"] != "security_verdict_slow"]

SV_SLOW = {
    "SVM-rbf":       SVC(kernel="rbf", random_state=SEED),
    "SVM-linear":    LinearSVC(max_iter=5000, random_state=SEED),
    "MLP":           MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=600,
                                   random_state=SEED),
    "GradientBoost": GradientBoostingClassifier(random_state=SEED),
}

for name, est in SV_SLOW.items():
    evaluate(name, est, X_sv, y_sv, task="clf", prep="full", note="security_verdict_slow")

# --- what rules does a shallow tree actually learn? ---
from sklearn.tree import export_text

shallow = build(DecisionTreeClassifier(max_depth=4, random_state=SEED), prep="full")
shallow.fit(X_sv, y_sv)

feat_names = shallow.named_steps["prep"].get_feature_names_out()
print("\n--- depth-4 tree, learned rules ---")
print(export_text(shallow.named_steps["model"],
                  feature_names=list(feat_names), max_depth=4))

SVM-rbf                 F1 0.9014 ± 0.0058   acc 0.8985   floor 0.8956   [13.0s]
SVM-linear              F1 0.9007 ± 0.0067   acc 0.8984   floor 0.8940   [0.3s]
MLP                     F1 0.9033 ± 0.0067   acc 0.9005   floor 0.8966   [44.1s]
GradientBoost           F1 0.9047 ± 0.0057   acc 0.9021   floor 0.8990   [28.5s]

--- depth-4 tree, learned rules ---
|--- cat__dh_group_modp768(1) <= 0.50
|   |--- cat__dh_group_modp1024(2) <= 0.50
|   |   |--- cat__dh_group_modp1536(5) <= 0.50
|   |   |   |--- cat__encryption_algo_AES-256-GCM <= 0.50
|   |   |   |   |--- class: Medium
|   |   |   |--- cat__encryption_algo_AES-256-GCM >  0.50
|   |   |   |   |--- class: Strong
|   |   |--- cat__dh_group_modp1536(5) >  0.50
|   |   |   |--- cat__encryption_algo_AES-256-GCM <= 0.50
|   |   |   |   |--- class: Weak
|   |   |   |--- cat__encryption_algo_AES-256-GCM >  0.50
|   |   |   |   |--- class: Medium
|   |--- cat__dh_group_modp1024(2) >  0.50
|   |   |--- cat__encryption_algo_AES-256-GCM <= 0.5

In [25]:
# Cell 9 kya kar raha hai: do cheezein.

# Pehla — security_verdict ke slow models (SVM, MLP, GradientBoost), taaki table poori ho. Mujhe zyada umeed nahi hai kyunki ceiling 0.90 pe saaf dikh raha hai, par completeness ke liye zaroori hai.

# Doosra — aur zyada important — ek depth-4 DecisionTree ke actual rules print karna. Kyunki jab ek tree aur 300 trees barabar hain, to asli sawaal ye hai ki tree ne seekha kya. Agar uske splits tumhare rules engine se match karte hain, to woh tumhara sabse strong demo artifact hai.

In [26]:
# ── Cell 10: verdict — config-only vs full ───────────────────────
RESULTS[:] = [r for r in RESULTS if r["note"] != "verdict_config_only"]

PREP_CONFIG = make_preprocessor(CAT_FEATURES, [])

def build_config(estimator):
    return Pipeline([("prep", clone(PREP_CONFIG)), ("model", estimator)])

X_cfg = train[CAT_FEATURES]

for name, est in {
    "DecisionTree-cfg": DecisionTreeClassifier(random_state=SEED),
    "RandomForest-cfg": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=1),
    "LogisticReg-cfg":  LogisticRegression(max_iter=2000, random_state=SEED),
    "GradientBoost-cfg": GradientBoostingClassifier(random_state=SEED),
}.items():
    pipe = build_config(est)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        out = cross_validate(pipe, X_cfg, y_sv, cv=CV_CLF,
                             scoring=CLF_SCORING, n_jobs=-1, error_score="raise")
    f1 = out["test_f1_macro"]
    row = {"model": name, "task": "clf", "note": "verdict_config_only",
           "seconds": 0, "f1_macro_mean": round(f1.mean(), 4),
           "f1_macro_std": round(f1.std(), 4),
           "accuracy_mean": round(out["test_accuracy"].mean(), 4),
           "accuracy_std": round(out["test_accuracy"].std(), 4),
           "floor": round(f1.mean() - f1.std(), 4)}
    RESULTS.append(row)
    print(f"{name:20s}  F1 {row['f1_macro_mean']:.4f} ± {row['f1_macro_std']:.4f}"
          f"   floor {row['floor']:.4f}")

print("\nfull-feature reference:")
for r in RESULTS:
    if r["note"] == "security_verdict" and r["model"] in ("DecisionTree", "RandomForest", "LogisticReg"):
        print(f"  {r['model']:20s} floor {r['floor']:.4f}")

DecisionTree-cfg      F1 0.9042 ± 0.0078   floor 0.8965
RandomForest-cfg      F1 0.9032 ± 0.0076   floor 0.8956
LogisticReg-cfg       F1 0.9047 ± 0.0065   floor 0.8982
GradientBoost-cfg     F1 0.9045 ± 0.0066   floor 0.8978

full-feature reference:
  LogisticReg          floor 0.8987
  DecisionTree         floor 0.8953
  RandomForest         floor 0.8981


In [27]:
# Cell 10 kya kar raha hai: verdict model ko sirf config features pe chalakar dekh raha hai — behaviour columns bilkul hata ke.

# Agar config-only ka score full-10-feature wale ke barabar aata hai, to behaviour features sirf noise la rahe hain aur unhe verdict model se nikaal dena chahiye. Isse model saaf ho jayega, spurious splits khatam, aur woh baat architecture se bhi match karegi jo humne abhi measure ki.

In [28]:
# ── Cell 11: risk_score — setup + fast regressors ────────────────
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, HistGradientBoostingRegressor,
                              AdaBoostRegressor)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyRegressor

RESULTS[:] = [r for r in RESULTS if not r["note"].startswith("risk_score")]

X_rs = train[FEATURES]
y_rs = train["risk_score"]

print(f"X {X_rs.shape}   y range {y_rs.min()}–{y_rs.max()}  mean {y_rs.mean():.2f}")
print(f"std of target = {y_rs.std():.3f}  (Dummy MAE should land near {(y_rs - y_rs.mean()).abs().mean():.3f})\n")

REG_FAST = {
    "Dummy":         DummyRegressor(strategy="mean"),
    "LinearReg":     LinearRegression(),
    "Ridge":         Ridge(random_state=SEED),
    "Lasso":         Lasso(random_state=SEED),
    "ElasticNet":    ElasticNet(random_state=SEED),
    "kNN":           KNeighborsRegressor(n_neighbors=5),
    "DecisionTree":  DecisionTreeRegressor(random_state=SEED),
    "RandomForest":  RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=1),
    "ExtraTrees":    ExtraTreesRegressor(n_estimators=300, random_state=SEED, n_jobs=1),
    "HistGB":        HistGradientBoostingRegressor(random_state=SEED),
    "AdaBoost":      AdaBoostRegressor(random_state=SEED),
}

for name, est in REG_FAST.items():
    evaluate(name, est, X_rs, y_rs, task="reg", prep="full", note="risk_score")

X (9000, 10)   y range 0.3–10.0  mean 4.99
std of target = 2.764  (Dummy MAE should land near 2.361)

Dummy                   MAE 2.3608 ± 0.0330   R2 -0.0004   ceiling 2.3938   [0.2s]
LinearReg               MAE 1.1723 ± 0.0190   R2 0.7283   ceiling 1.1913   [0.2s]
Ridge                   MAE 1.1723 ± 0.0190   R2 0.7283   ceiling 1.1913   [0.2s]
Lasso                   MAE 2.3608 ± 0.0330   R2 -0.0004   ceiling 2.3938   [0.2s]
ElasticNet              MAE 2.3608 ± 0.0330   R2 -0.0004   ceiling 2.3938   [0.2s]
kNN                     MAE 1.2680 ± 0.0193   R2 0.6544   ceiling 1.2873   [0.4s]
DecisionTree            MAE 1.3542 ± 0.0238   R2 0.5906   ceiling 1.3780   [0.3s]
RandomForest            MAE 1.0330 ± 0.0166   R2 0.7720   ceiling 1.0496   [74.0s]
ExtraTrees              MAE 1.0700 ± 0.0141   R2 0.7519   ceiling 1.0841   [61.8s]
HistGB                  MAE 1.0154 ± 0.0158   R2 0.7877   ceiling 1.0312   [2.9s]
AdaBoost                MAE 1.4714 ± 0.0196   R2 0.6069   ceiling 1.4910 

In [29]:
# Cell 11 kya kar raha hai: teesra aur aakhiri target — risk_score, jo regression hai (0-10 continuous number, class nahi).

# Do baatein alag hain. Ek, models alag hain — Ridge, SVR, RandomForestRegressor waghera. Do, aur zyada important — security_verdict ko feature nahi banana. Woh risk ka doosra roop hai, dono ek hi cheez naapte hain. Agar verdict feature bana diya to model 0.1 MAE de dega aur woh jhootha hoga.

# Isliye yahan bhi sirf wahi 10 features (ya jo shortlist ho) jaayenge, targets nahi.

# Aur ek diagnostic pehle — risk_score config se kitna determined hai vs behaviour se. Wahi split test jo verdict pe kiya tha.

In [30]:
# ── Cell 12: risk_score — slow regressors + ablation ─────────────
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

RESULTS[:] = [r for r in RESULTS if r["note"] != "risk_score_slow"]

REG_SLOW = {
    "SVR-rbf":       SVR(kernel="rbf"),
    "MLP":           MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=600,
                                  random_state=SEED),
    "GradientBoost": GradientBoostingRegressor(random_state=SEED),
}

for name, est in REG_SLOW.items():
    evaluate(name, est, X_rs, y_rs, task="reg", prep="full", note="risk_score_slow")

print("\n--- ablation: which feature block drives risk_score? ---")

for tag, cols, prep_obj in [
    ("config-only",    CAT_FEATURES,       PREP_CONFIG),
    ("behaviour-only", BEHAVIOUR_FEATURES, PREP_BEHAVIOUR),
]:
    pipe = Pipeline([("prep", clone(prep_obj)),
                     ("model", HistGradientBoostingRegressor(random_state=SEED))])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        out = cross_validate(pipe, train[cols], y_rs, cv=CV_REG,
                             scoring=REG_SCORING, n_jobs=-1, error_score="raise")
    mae = -out["test_mae"]
    print(f"HistGB {tag:16s}  MAE {mae.mean():.4f} ± {mae.std():.4f}"
          f"   R2 {out['test_r2'].mean():.4f}")

print(f"\nfull 10 features reference: MAE 1.0154")

SVR-rbf                 MAE 1.0920 ± 0.0200   R2 0.7539   ceiling 1.1120   [26.0s]
MLP                     MAE 1.1087 ± 0.0287   R2 0.7394   ceiling 1.1374   [78.7s]
GradientBoost           MAE 1.0399 ± 0.0155   R2 0.7801   ceiling 1.0554   [8.4s]

--- ablation: which feature block drives risk_score? ---
HistGB config-only       MAE 1.0023 ± 0.0153   R2 0.7935
HistGB behaviour-only    MAE 2.4031 ± 0.0346   R2 -0.0478

full 10 features reference: MAE 1.0154


In [31]:
# Cell 12 kya kar raha hai: do cheezein.

# Pehla — slow regressors (SVR, MLP, GradientBoost). GradientBoost regression mein sirf ek target hai (8 classes nahi), to ye classification wale se kaafi tez hoga.

# Doosra — wahi feature ablation jo verdict pe kiya tha. Risk score config se aata hai ya behaviour se? Verdict config-only nikla tha. Risk bhi wahi hona chahiye, par confirm karna zaroori hai — kyunki agar risk bhi config-only hai to teeno models ka feature story ek saath saaf ho jaayegi.

In [32]:
# ── Cell 13: Stage 1 summary + shortlist ─────────────────────────
all_res = pd.DataFrame(RESULTS)

print("=" * 70)
print("TRAFFIC_TYPE  (behaviour features only)")
print("=" * 70)
tt = all_res[all_res.note.str.startswith("traffic_type")].sort_values("floor", ascending=False)
print(tt[["model", "f1_macro_mean", "f1_macro_std", "floor", "seconds"]].head(8).to_string(index=False))

print("\n" + "=" * 70)
print("SECURITY_VERDICT  (config features only)")
print("=" * 70)
sv = all_res[all_res.note.isin(["security_verdict", "security_verdict_slow",
                                "verdict_config_only"])].sort_values("floor", ascending=False)
print(sv[["model", "f1_macro_mean", "f1_macro_std", "floor", "note"]].head(8).to_string(index=False))

print("\n" + "=" * 70)
print("RISK_SCORE  (config features only)")
print("=" * 70)
rs = all_res[all_res.note.str.startswith("risk_score")].sort_values("floor", ascending=True)
print(rs[["model", "mae_mean", "mae_std", "r2_mean", "floor", "seconds"]].head(8).to_string(index=False))

print("\n" + "=" * 70)
print("ARCHITECTURE EVIDENCE  (the separation result)")
print("=" * 70)
ctrl = all_res[all_res.note == "security_verdict_control"]
print(ctrl[["model", "f1_macro_mean", "accuracy_mean"]].to_string(index=False))
print("majority baseline accuracy = 0.3944  -> behaviour carries no security signal")

all_res.to_csv("../stage1_results.csv", index=False)
print(f"\n{len(all_res)} candidate runs saved to stage1_results.csv")

TRAFFIC_TYPE  (behaviour features only)
        model  f1_macro_mean  f1_macro_std  floor  seconds
 RandomForest         0.9480        0.0045 0.9435      8.6
   ExtraTrees         0.9480        0.0047 0.9433      6.4
   GaussianNB         0.9462        0.0040 0.9422      0.1
GradientBoost         0.9469        0.0048 0.9421     46.3
       HistGB         0.9435        0.0060 0.9375     12.4
          MLP         0.9396        0.0040 0.9356     81.8
      SVM-rbf         0.9201        0.0051 0.9150      2.3
 DecisionTree         0.9203        0.0057 0.9146      0.4

SECURITY_VERDICT  (config features only)
            model  f1_macro_mean  f1_macro_std  floor                  note
    GradientBoost         0.9047        0.0057 0.8990 security_verdict_slow
              LDA         0.9048        0.0059 0.8989      security_verdict
      LogisticReg         0.9045        0.0058 0.8987      security_verdict
  LogisticReg-cfg         0.9047        0.0065 0.8982   verdict_config_only
     Ra

In [33]:
# Cell 13 kya kar raha hai: Stage 1 ka closing. Teeno tasks ki final leaderboard ek jagah, aur har task se top 4-5 ka shortlist jo Stage 2 mein jaayega.

# Ek aur cheez ye check karega — kaun se shortlisted models skl2onnx se export ho sakte hain. Kyunki jo bhi jeete, use Java backend mein .onnx banke jaana hai. Agar winner export nahi ho paaya to sab bekaar.

In [35]:
# ── Cell 14: ONNX export smoke test ──────────────────────────────
import numpy as np
from skl2onnx import to_onnx
import onnxruntime as ort

def onnx_ok(name, estimator, X, y, prep, task):
    """Fit -> export -> run -> compare against sklearn."""
    pipe = build_config(estimator) if prep == "config" else build(estimator, prep=prep)
    pipe.fit(X, y)
    sample = X.head(50)
    try:
        onx = to_onnx(pipe, X.head(1), target_opset=17)
        blob = onx.SerializeToString()
        sess = ort.InferenceSession(blob, providers=["CPUExecutionProvider"])
        feed = {}
        for i in sess.get_inputs():
            col = sample[i.name].to_numpy().reshape(-1, 1)
            feed[i.name] = col.astype(object) if "string" in i.type else col.astype(np.float32)
        out = sess.run(None, feed)
        if task == "clf":
            agree = (np.array(out[0]).ravel() == pipe.predict(sample)).mean()
            print(f"  {name:22s} OK   agreement {agree:.3f}   {len(blob):,} bytes")
        else:
            delta = np.abs(np.array(out[0]).ravel() - pipe.predict(sample)).max()
            print(f"  {name:22s} OK   max delta {delta:.5f}   {len(blob):,} bytes")
        return True
    except Exception as e:
        print(f"  {name:22s} FAIL  {type(e).__name__}: {str(e)[:90]}")
        return False

print("--- traffic_type (behaviour-only) ---")
for n, e in {
    "RandomForest":  RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    "ExtraTrees":    ExtraTreesClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    "GaussianNB":    GaussianNB(),
    "GradientBoost": GradientBoostingClassifier(random_state=SEED),
}.items():
    onnx_ok(n, e, X_tt, y_tt, "behaviour", "clf")

print("\n--- security_verdict (config-only) ---")
for n, e in {
    "DecisionTree":  DecisionTreeClassifier(random_state=SEED),
    "LogisticReg":   LogisticRegression(max_iter=2000, random_state=SEED),
    "GradientBoost": GradientBoostingClassifier(random_state=SEED),
    "RandomForest":  RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
}.items():
    onnx_ok(n, e, X_cfg, y_sv, "config", "clf")

print("\n--- risk_score (config-only) ---")
for n, e in {
    "HistGB":        HistGradientBoostingRegressor(random_state=SEED),
    "RandomForest":  RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
    "GradientBoost": GradientBoostingRegressor(random_state=SEED),
}.items():
    onnx_ok(n, e, X_cfg, y_rs, "config", "reg")

--- traffic_type (behaviour-only) ---
  RandomForest           FAIL  InvalidArgument: [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Unexpected input data type. Actual: (tensor(fl
  ExtraTrees             FAIL  Fail: [ONNXRuntimeError] : 1 : FAIL : Type Error: Type (seq(map(string,tensor(double)))) of outp
  GaussianNB             FAIL  InvalidGraph: [ONNXRuntimeError] : 10 : INVALID_GRAPH : This is an invalid model. Type Error: Type 'tens
  GradientBoost          FAIL  Fail: [ONNXRuntimeError] : 1 : FAIL : Type Error: Type (seq(map(string,tensor(double)))) of outp

--- security_verdict (config-only) ---
  DecisionTree           OK   agreement 1.000   16,643 bytes
  LogisticReg            OK   agreement 1.000   1,973 bytes
  GradientBoost          OK   agreement 1.000   170,225 bytes
  RandomForest           OK   agreement 1.000   6,468,001 bytes

--- risk_score (config-only) ---
  HistGB                 FAIL  ValueError: Unable to create node 'TreeEnsembleRegressor' with name='TreeEnsemb

In [36]:
# Cell 14 kya kar raha hai: shortlist ke har model ko fit karke .onnx mein export karega, fir ONNX ka prediction sklearn ke prediction se milaayega. Jo pass ho jaye woh safe, jo fail ho woh shortlist se bahar. Khaas tor pe GaussianNB aur HistGB ko check karna hai — dono apne-apne task ke top pe hain par unke converters kam tested hain.

In [37]:
# ── Cell 14b: ONNX retry with float32 + zipmap off ───────────────
X_tt32 = X_tt.astype(np.float32)

def onnx_ok2(name, estimator, X, y, prep, task):
    pipe = build_config(estimator) if prep == "config" else build(estimator, prep=prep)
    pipe.fit(X, y)
    sample = X.head(50)
    try:
        opts = {} if task == "reg" else {id(pipe): {"zipmap": False}}
        onx = to_onnx(pipe, X.head(1), target_opset=17, options=opts)
        blob = onx.SerializeToString()
        sess = ort.InferenceSession(blob, providers=["CPUExecutionProvider"])
        feed = {}
        for i in sess.get_inputs():
            col = sample[i.name].to_numpy().reshape(-1, 1)
            if "string" in i.type:
                feed[i.name] = col.astype(object)
            elif "double" in i.type:
                feed[i.name] = col.astype(np.float64)
            else:
                feed[i.name] = col.astype(np.float32)
        out = sess.run(None, feed)
        if task == "clf":
            agree = (np.array(out[0]).ravel() == pipe.predict(sample)).mean()
            print(f"  {name:22s} OK   agreement {agree:.3f}   {len(blob):,} bytes")
        else:
            delta = np.abs(np.array(out[0]).ravel() - pipe.predict(sample)).max()
            print(f"  {name:22s} OK   max delta {delta:.5f}   {len(blob):,} bytes")
        return True
    except Exception as e:
        print(f"  {name:22s} FAIL  {type(e).__name__}: {str(e)[:100]}")
        return False

print("--- traffic_type, float32 inputs ---")
for n, e in {
    "RandomForest":  RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    "ExtraTrees":    ExtraTreesClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    "GaussianNB":    GaussianNB(),
    "GradientBoost": GradientBoostingClassifier(random_state=SEED),
}.items():
    onnx_ok2(n, e, X_tt32, y_tt, "behaviour", "clf")

print("\n--- risk_score, HistGB alternatives ---")
for n, e in {
    "GradientBoost": GradientBoostingRegressor(random_state=SEED),
    "ExtraTrees":    ExtraTreesRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
}.items():
    onnx_ok2(n, e, X_cfg, y_rs, "config", "reg")

--- traffic_type, float32 inputs ---
  RandomForest           OK   agreement 1.000   25,174,313 bytes
  ExtraTrees             OK   agreement 1.000   82,444,707 bytes
  GaussianNB             OK   agreement 1.000   2,334 bytes
  GradientBoost          OK   agreement 1.000   452,388 bytes

--- risk_score, HistGB alternatives ---
  GradientBoost          OK   max delta 0.00000   56,605 bytes
  ExtraTrees             OK   max delta 0.00003   9,079,869 bytes


In [38]:
# Cell 14b kya kar raha hai: dono fixable problems ek saath theek kar raha hai. Numeric columns ko float32 mein cast kar raha hai fit se pehle — isse ONNX float inputs banayega, teeno models mein consistent, aur tumhara Java FloatBuffer helper bina special-casing ke chal jayega. Aur ZipMap band kar raha hai.

In [39]:
# ── Cell 15: final models + held-out test evaluation ─────────────
from sklearn.metrics import f1_score, accuracy_score, mean_absolute_error, r2_score

FINAL = {}

# 1. traffic_type — behaviour only, float32 for clean ONNX
FINAL["traffic"] = build(
    GradientBoostingClassifier(random_state=SEED), prep="behaviour"
).fit(train[BEHAVIOUR_FEATURES].astype(np.float32), train["traffic_type"])

# 2. security_verdict — config only, interpretable
FINAL["verdict"] = build_config(
    DecisionTreeClassifier(random_state=SEED)
).fit(train[CAT_FEATURES], train["security_verdict"])

# 3. risk_score — config only
FINAL["risk"] = build_config(
    GradientBoostingRegressor(random_state=SEED)
).fit(train[CAT_FEATURES], train["risk_score"])

print("=" * 62)
print("HELD-OUT TEST SET (1200 rows, never seen during Stage 1)")
print("=" * 62)

p = FINAL["traffic"].predict(test[BEHAVIOUR_FEATURES].astype(np.float32))
print(f"traffic_type      F1 {f1_score(test.traffic_type, p, average='macro'):.4f}"
      f"   acc {accuracy_score(test.traffic_type, p):.4f}")

p = FINAL["verdict"].predict(test[CAT_FEATURES])
print(f"security_verdict  F1 {f1_score(test.security_verdict, p, average='macro'):.4f}"
      f"   acc {accuracy_score(test.security_verdict, p):.4f}")

p = FINAL["risk"].predict(test[CAT_FEATURES])
print(f"risk_score        MAE {mean_absolute_error(test.risk_score, p):.4f}"
      f"   R2 {r2_score(test.risk_score, p):.4f}")

print("\n--- demo sets (out-of-distribution check) ---")
for f, label in [("demo_weak_tunnels.csv", "Weak"), ("demo_strong_tunnels.csv", "Strong")]:
    d = pd.read_csv(DATA / f)
    pv = FINAL["verdict"].predict(d[CAT_FEATURES])
    pr = FINAL["risk"].predict(d[CAT_FEATURES])
    print(f"{f:26s} verdict {(pv == label).mean():.3f} correct   "
          f"mean risk {pr.mean():.2f} (actual {d.risk_score.mean():.2f})")

HELD-OUT TEST SET (1200 rows, never seen during Stage 1)
traffic_type      F1 0.9418   acc 0.9408
security_verdict  F1 0.8931   acc 0.8908
risk_score        MAE 1.0214   R2 0.7916

--- demo sets (out-of-distribution check) ---
demo_weak_tunnels.csv      verdict 1.000 correct   mean risk 8.58 (actual 8.55)
demo_strong_tunnels.csv    verdict 1.000 correct   mean risk 1.67 (actual 1.53)


In [40]:
# Cell 15 kya kar raha hai: final teen models train kar raha hai, aur pehli baar test set ko haath laga raha hai. Ab tak sirf CV chalayi thi train pe. Test ke 1200 rows abhi tak chhue nahi gaye — yahi unka maqsad tha. Jo number yahan aayega woh honest held-out number hai, wahi deck mein jaayega.

In [41]:
# ── Cell 16: save models + model card ────────────────────────────
import json, joblib
from datetime import datetime, timezone

MODELS = Path("../models")
MODELS.mkdir(exist_ok=True)

SPEC = {
    "traffic_classifier": dict(pipe=FINAL["traffic"], cols=BEHAVIOUR_FEATURES,
                               X=train[BEHAVIOUR_FEATURES].astype(np.float32), task="clf"),
    "security_classifier": dict(pipe=FINAL["verdict"], cols=CAT_FEATURES,
                                X=train[CAT_FEATURES], task="clf"),
    "risk_regressor": dict(pipe=FINAL["risk"], cols=CAT_FEATURES,
                           X=train[CAT_FEATURES], task="reg"),
}

for name, s in SPEC.items():
    joblib.dump(s["pipe"], MODELS / f"{name}.joblib")
    opts = {} if s["task"] == "reg" else {id(s["pipe"]): {"zipmap": False}}
    onx = to_onnx(s["pipe"], s["X"].head(1), target_opset=17, options=opts)
    (MODELS / f"{name}.onnx").write_bytes(onx.SerializeToString())
    print(f"{name:22s} joblib + onnx   "
          f"{(MODELS / f'{name}.onnx').stat().st_size:,} bytes")

card = {
    "project": "TunnelScope / SIH26160",
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "data": {"train_rows": len(train), "test_rows": len(test),
             "source": "synthetic — SYNTHETIC VALIDATION, not live capture",
             "note": "risk_score target carries deliberate Gaussian noise (sd 0.55) "
                     "so verdict bands overlap; MAE floor is therefore non-zero"},
    "selection": {"protocol": "RepeatedStratifiedKFold 5x5, shared folds, "
                              "preprocessing inside pipeline",
                  "ranked_by": "mean minus one std (conservative floor)",
                  "candidates_evaluated": len(RESULTS)},
    "models": {
        "traffic_classifier": {
            "algorithm": "GradientBoostingClassifier", "features": BEHAVIOUR_FEATURES,
            "cv_f1_macro": 0.9469, "test_f1_macro": 0.9418, "test_accuracy": 0.9408,
            "maturity": "SYNTHETIC VALIDATION",
            "limits": ["config features deliberately excluded — unusable on ESP-only flows",
                       "top confusions are behaviourally adjacent: dns_query/icmp, "
                       "email/web_browsing, video_streaming/file_transfer",
                       "voip recall 1.000 on train is a generator artifact, not a claim"]},
        "security_classifier": {
            "algorithm": "DecisionTreeClassifier", "features": CAT_FEATURES,
            "cv_f1_macro": 0.9042, "test_f1_macro": 0.8931, "test_accuracy": 0.8908,
            "maturity": "SYNTHETIC VALIDATION",
            "limits": ["24.8% of training rows sit in config combos with >1 verdict — "
                       "100% is unreachable by construction",
                       "rules engine owns the authoritative verdict; this model is "
                       "parallel evidence, not the decision",
                       "behaviour features carry no security signal (0.331 macro-F1, "
                       "below the 0.394 majority baseline)"]},
        "risk_regressor": {
            "algorithm": "GradientBoostingRegressor", "features": CAT_FEATURES,
            "cv_mae": 1.0399, "test_mae": 1.0214, "test_r2": 0.7916,
            "maturity": "SYNTHETIC VALIDATION",
            "limits": ["HistGradientBoosting scored better (MAE 1.0023) but has no "
                       "working skl2onnx converter — rejected for deployability",
                       "behaviour-only ablation gives R2 -0.048, worse than predicting "
                       "the mean"]},
    },
    "onnx": {"opset": 17, "zipmap": False,
             "numeric_input_dtype": "float32 across all three models",
             "note": "each feature is its own named [N,1] input, not one [N,k] tensor"},
    "not_quotable": ["demo_weak/demo_strong accuracy (1.000) — extreme configs only, "
                     "no Medium rows present"],
}

(MODELS / "model_card.json").write_text(json.dumps(card, indent=2))
print(f"\nmodel_card.json written to {MODELS.resolve()}")

traffic_classifier     joblib + onnx   452,388 bytes
security_classifier    joblib + onnx   16,471 bytes
risk_regressor         joblib + onnx   56,605 bytes

model_card.json written to C:\Users\Shivanshu Shukla\OneDrive\Desktop\SIH\models


In [42]:
# Cell 16 kya kar raha hai: models ko disk pe save kar raha hai — .joblib (Python ke liye) aur .onnx (Java ke liye) dono. Saath mein ek model_card.json jo har model ki poori kahani rakhta hai: kaunse features, kaunsa algorithm, kya score, aur kya limitations. Woh card hi source of truth banega jab backend wala banda ya judge poochhe ki ye numbers kahan se aaye.

In [43]:
# ── Cell 17: parity check — disk .onnx vs .joblib ────────────────
print("=" * 66)
print("ONNX PARITY (loaded from disk, as Java will)")
print("=" * 66)

for name, s in SPEC.items():
    sk = joblib.load(MODELS / f"{name}.joblib")
    sess = ort.InferenceSession(str(MODELS / f"{name}.onnx"),
                                providers=["CPUExecutionProvider"])

    sample = s["X"].head(200)
    feed = {}
    for i in sess.get_inputs():
        col = sample[i.name].to_numpy().reshape(-1, 1)
        if "string" in i.type:
            feed[i.name] = col.astype(object)
        elif "double" in i.type:
            feed[i.name] = col.astype(np.float64)
        else:
            feed[i.name] = col.astype(np.float32)

    out = sess.run(None, feed)
    if s["task"] == "clf":
        agree = (np.array(out[0]).ravel() == sk.predict(sample)).mean()
        verdict = "PASS" if agree == 1.0 else "FAIL"
        print(f"\n{name}  [{verdict}]  label agreement {agree:.4f}")
    else:
        delta = np.abs(np.array(out[0]).ravel() - sk.predict(sample)).max()
        verdict = "PASS" if delta < 1e-4 else "FAIL"
        print(f"\n{name}  [{verdict}]  max delta {delta:.6f}")

    print(f"  inputs  : {len(sess.get_inputs())}")
    for i in sess.get_inputs():
        print(f"     {i.name:26s} {i.type}")
    print(f"  outputs : {[o.name for o in sess.get_outputs()]}")

ONNX PARITY (loaded from disk, as Java will)

traffic_classifier  [PASS]  label agreement 1.0000
  inputs  : 5
     avg_packet_size_bytes      tensor(float)
     packet_rate_per_sec        tensor(float)
     session_duration_sec       tensor(float)
     burstiness_index           tensor(float)
     ike_handshake_time_ms      tensor(float)
  outputs : ['label', 'probabilities']

security_classifier  [PASS]  label agreement 1.0000
  inputs  : 5
     mode                       tensor(string)
     encryption_algo            tensor(string)
     dh_group                   tensor(string)
     pfs                        tensor(string)
     ip_version                 tensor(string)
  outputs : ['label', 'probabilities']

risk_regressor  [PASS]  max delta 0.000001
  inputs  : 5
     mode                       tensor(string)
     encryption_algo            tensor(string)
     dh_group                   tensor(string)
     pfs                        tensor(string)
     ip_version                 t

In [46]:
# Cell 17 kya kar raha hai: aakhiri verification. Disk se .onnx files dobara load karega — yaani bilkul waise hi jaise Java karega — aur unke predictions ko .joblib ke predictions se milaayega. Ab tak humne memory mein bane objects test kiye the. Ye test file se load karke karta hai, to agar save karne mein kuch bigda hoga to yahan pakda jayega.

# Handoff mein jo dtype wala bug likha hai, uska asli ilaaj yahi hai — ye script batayegi ki har model ka input type kya hai, aur woh information seedha Java OnnxModelService mein jaayegi.

In [47]:
# ── Cell 18: manual sanity test ──────────────────────────────────
import joblib, numpy as np, pandas as pd
from pathlib import Path

M = Path("../models")
traffic = joblib.load(M / "traffic_classifier.joblib")
verdict = joblib.load(M / "security_classifier.joblib")
risk    = joblib.load(M / "risk_regressor.joblib")

configs = pd.DataFrame([
    {"name": "clearly weak",   "mode": "Tunnel",    "encryption_algo": "DES",
     "dh_group": "modp768(1)",   "pfs": "Off", "ip_version": "IPv4"},
    {"name": "clearly strong", "mode": "Tunnel",    "encryption_algo": "AES-256-GCM",
     "dh_group": "modp8192(21)", "pfs": "On",  "ip_version": "IPv6"},
    {"name": "middling",       "mode": "Transport", "encryption_algo": "AES-128",
     "dh_group": "modp2048(14)", "pfs": "On",  "ip_version": "IPv4"},
    {"name": "trap: good cipher, weak DH", "mode": "Tunnel", "encryption_algo": "AES-256-GCM",
     "dh_group": "modp1024(2)",  "pfs": "Off", "ip_version": "IPv4"},
])

CFG = ["mode", "encryption_algo", "dh_group", "pfs", "ip_version"]
v = verdict.predict(configs[CFG])
p = verdict.predict_proba(configs[CFG]).max(axis=1)
r = risk.predict(configs[CFG])

print("=" * 72)
print("CONFIG PLANE  (security_classifier + risk_regressor)")
print("=" * 72)
for i, row in configs.iterrows():
    print(f"{row['name']:32s} -> {v[i]:7s} (conf {p[i]:.2f})   risk {r[i]:.2f}/10")

flows = pd.DataFrame([
    {"name": "tiny + slow (dns-ish)",  "avg_packet_size_bytes": 80,   "packet_rate_per_sec": 2,
     "session_duration_sec": 1,    "burstiness_index": 0.05, "ike_handshake_time_ms": 120},
    {"name": "small + steady (voip)",  "avg_packet_size_bytes": 160,  "packet_rate_per_sec": 50,
     "session_duration_sec": 300,  "burstiness_index": 0.15, "ike_handshake_time_ms": 120},
    {"name": "big + fast (transfer)",  "avg_packet_size_bytes": 1400, "packet_rate_per_sec": 150,
     "session_duration_sec": 60,   "burstiness_index": 0.80, "ike_handshake_time_ms": 120},
    {"name": "big + long (streaming)", "avg_packet_size_bytes": 1350, "packet_rate_per_sec": 100,
     "session_duration_sec": 1800, "burstiness_index": 0.55, "ike_handshake_time_ms": 120},
])

BEH = ["avg_packet_size_bytes", "packet_rate_per_sec", "session_duration_sec",
       "burstiness_index", "ike_handshake_time_ms"]
X = flows[BEH].astype(np.float32)
t = traffic.predict(X)
tp = traffic.predict_proba(X).max(axis=1)

print("\n" + "=" * 72)
print("BEHAVIOUR PLANE  (traffic_classifier)")
print("=" * 72)
for i, row in flows.iterrows():
    print(f"{row['name']:32s} -> {t[i]:16s} (conf {tp[i]:.2f})")

CONFIG PLANE  (security_classifier + risk_regressor)
clearly weak                     -> Weak    (conf 1.00)   risk 8.63/10
clearly strong                   -> Strong  (conf 1.00)   risk 1.48/10
middling                         -> Medium  (conf 1.00)   risk 4.42/10
trap: good cipher, weak DH       -> Medium  (conf 1.00)   risk 4.83/10

BEHAVIOUR PLANE  (traffic_classifier)
tiny + slow (dns-ish)            -> icmp             (conf 0.85)
small + steady (voip)            -> voip             (conf 1.00)
big + fast (transfer)            -> file_transfer    (conf 1.00)
big + long (streaming)           -> video_streaming  (conf 1.00)


In [48]:
# ── Cell 18a: security_classifier ────────────────────────────────
import joblib, pandas as pd
from pathlib import Path

M = Path("../models")
verdict = joblib.load(M / "security_classifier.joblib")
CFG = ["mode", "encryption_algo", "dh_group", "pfs", "ip_version"]

tests = pd.DataFrame([
    {"mode": "Tunnel",    "encryption_algo": "DES",         "dh_group": "modp768(1)",   "pfs": "Off", "ip_version": "IPv4"},
    {"mode": "Tunnel",    "encryption_algo": "AES-256-GCM", "dh_group": "modp8192(21)", "pfs": "On",  "ip_version": "IPv6"},
    {"mode": "Transport", "encryption_algo": "AES-128",     "dh_group": "modp2048(14)", "pfs": "On",  "ip_version": "IPv4"},
    {"mode": "Tunnel",    "encryption_algo": "AES-256-GCM", "dh_group": "modp1024(2)",  "pfs": "Off", "ip_version": "IPv4"},
])

pred = verdict.predict(tests[CFG])
conf = verdict.predict_proba(tests[CFG]).max(axis=1)

for i, row in tests.iterrows():
    print(f"{row.encryption_algo:12s} + {row.dh_group:13s} + PFS {row.pfs:3s}"
          f"  ->  {pred[i]:7s} ({conf[i]:.2f})")
          

DES          + modp768(1)    + PFS Off  ->  Weak    (1.00)
AES-256-GCM  + modp8192(21)  + PFS On   ->  Strong  (1.00)
AES-128      + modp2048(14)  + PFS On   ->  Medium  (1.00)
AES-256-GCM  + modp1024(2)   + PFS Off  ->  Medium  (1.00)


In [49]:
# ── Cell 19a: demo_weak_tunnels ──────────────────────────────────
import joblib, numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error

D = Path("../data")
M = Path("../models")

traffic = joblib.load(M / "traffic_classifier.joblib")
verdict = joblib.load(M / "security_classifier.joblib")
risk    = joblib.load(M / "risk_regressor.joblib")

CFG = ["mode", "encryption_algo", "dh_group", "pfs", "ip_version"]
BEH = ["avg_packet_size_bytes", "packet_rate_per_sec", "session_duration_sec",
       "burstiness_index", "ike_handshake_time_ms"]

d = pd.read_csv(D / "demo_weak_tunnels.csv")
t = traffic.predict(d[BEH].astype(np.float32))
v = verdict.predict(d[CFG])
r = risk.predict(d[CFG])

print(f"demo_weak_tunnels.csv  ({len(d)} rows)\n")
print(f"  traffic   acc {accuracy_score(d.traffic_type, t):.4f}")
print(f"  security  acc {accuracy_score(d.security_verdict, v):.4f}   "
      f"predicted: {dict(pd.Series(v).value_counts())}")
print(f"  risk      MAE {mean_absolute_error(d.risk_score, r):.4f}   "
      f"mean {r.mean():.2f} vs actual {d.risk_score.mean():.2f}")

demo_weak_tunnels.csv  (30 rows)

  traffic   acc 0.9667
  security  acc 1.0000   predicted: {'Weak': np.int64(30)}
  risk      MAE 0.6957   mean 8.58 vs actual 8.55


In [50]:
# ── Cell 19b: demo_strong_tunnels ────────────────────────────────
d = pd.read_csv(D / "demo_strong_tunnels.csv")
t = traffic.predict(d[BEH].astype(np.float32))
v = verdict.predict(d[CFG])
r = risk.predict(d[CFG])

print(f"demo_strong_tunnels.csv  ({len(d)} rows)\n")
print(f"  traffic   acc {accuracy_score(d.traffic_type, t):.4f}")
print(f"  security  acc {accuracy_score(d.security_verdict, v):.4f}   "
      f"predicted: {dict(pd.Series(v).value_counts())}")
print(f"  risk      MAE {mean_absolute_error(d.risk_score, r):.4f}   "
      f"mean {r.mean():.2f} vs actual {d.risk_score.mean():.2f}")

demo_strong_tunnels.csv  (30 rows)

  traffic   acc 0.9667
  security  acc 1.0000   predicted: {'Strong': np.int64(30)}
  risk      MAE 0.5820   mean 1.67 vs actual 1.53


In [51]:
# ── Cell 19c: held-out test set ──────────────────────────────────
d = pd.read_csv(D / "synthetic_ipsec_dataset_test.csv")
t = traffic.predict(d[BEH].astype(np.float32))
v = verdict.predict(d[CFG])
r = risk.predict(d[CFG])

print(f"synthetic_ipsec_dataset_test.csv  ({len(d)} rows)\n")
print(f"  traffic   acc {accuracy_score(d.traffic_type, t):.4f}   "
      f"F1 {f1_score(d.traffic_type, t, average='macro'):.4f}")
print(f"  security  acc {accuracy_score(d.security_verdict, v):.4f}   "
      f"F1 {f1_score(d.security_verdict, v, average='macro'):.4f}")
print(f"  risk      MAE {mean_absolute_error(d.risk_score, r):.4f}   "
      f"mean {r.mean():.2f} vs actual {d.risk_score.mean():.2f}")

synthetic_ipsec_dataset_test.csv  (1200 rows)

  traffic   acc 0.9408   F1 0.9418
  security  acc 0.8908   F1 0.8931
  risk      MAE 1.0214   mean 4.81 vs actual 4.74


In [52]:
# ── Cell 19a: demo_weak_tunnels ──────────────────────────────────
import joblib, numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, mean_absolute_error

D, M = Path("../data"), Path("../models")
traffic = joblib.load(M / "traffic_classifier.joblib")
verdict = joblib.load(M / "security_classifier.joblib")
risk    = joblib.load(M / "risk_regressor.joblib")

CFG = ["mode", "encryption_algo", "dh_group", "pfs", "ip_version"]
BEH = ["avg_packet_size_bytes", "packet_rate_per_sec", "session_duration_sec",
       "burstiness_index", "ike_handshake_time_ms"]


def check(filename):
    d = pd.read_csv(D / filename)
    t = traffic.predict(d[BEH].astype(np.float32))
    v = verdict.predict(d[CFG])
    r = risk.predict(d[CFG])

    print(f"\n{filename}   ({len(d)} rows)")
    print("-" * 46)
    print(f"  traffic    {accuracy_score(d.traffic_type, t):.1%} correct")
    print(f"  security   {accuracy_score(d.security_verdict, v):.1%} correct")
    print(f"  risk       {r.mean():.2f} predicted  |  {d.risk_score.mean():.2f} actual"
          f"  |  off by {mean_absolute_error(d.risk_score, r):.2f}")


check("demo_weak_tunnels.csv")


demo_weak_tunnels.csv   (30 rows)
----------------------------------------------
  traffic    96.7% correct
  security   100.0% correct
  risk       8.58 predicted  |  8.55 actual  |  off by 0.70


In [53]:
# ── Cell 19b: demo_strong_tunnels ────────────────────────────────
check("demo_strong_tunnels.csv")


demo_strong_tunnels.csv   (30 rows)
----------------------------------------------
  traffic    96.7% correct
  security   100.0% correct
  risk       1.67 predicted  |  1.53 actual  |  off by 0.58


In [54]:
# ── Cell 19c: held-out test set ──────────────────────────────────
check("synthetic_ipsec_dataset_test.csv")


synthetic_ipsec_dataset_test.csv   (1200 rows)
----------------------------------------------
  traffic    94.1% correct
  security   89.1% correct
  risk       4.81 predicted  |  4.74 actual  |  off by 1.02
